# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Подготовка данных

In [19]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, folder_override=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.folder_override = folder_override
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        file_name = self.data.iloc[idx]['file_name']
        if self.folder_override:
            file_name = self.folder_override + '/' + file_name.split('/')[-1]
        img_path = self.root_dir / file_name
        image = Image.open(img_path).convert('RGB')
        label = int(self.data.iloc[idx]['label'])
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [20]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file=f'{BASE_DIR}/ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir=f'{BASE_DIR}/ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file=f'{BASE_DIR}/ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir=f'{BASE_DIR}/ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform,
    folder_override='test_data'
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 9993
Test dataset size: 3997


## 3. Создание модели ResNet18

In [5]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s] 


Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): R

## 4. Определение функции потерь и оптимизатора

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [12]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    # добавьте подсчет epoch_loss, epoch_acc, epoch_f1

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [13]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # добавьте подсчет epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted')
    epoch_recall = recall_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [14]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir='runs/train_1')

num_epochs = 10

writer.add_text('hyperparameters', 
    f'lr=0.001, batch_size={batch_size}, epochs={num_epochs}, optimizer=Adam, scheduler=StepLR(step_size=5, gamma=0.1)')

train_losses = []
train_accs = []
train_f1s = []

# Добавьте логгирование метрик для tensorboard

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)
    
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)

    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('F1/train', train_f1, epoch)
    
    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')

print('\nTraining completed!')


Epoch 1/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:51<00:00,  6.07it/s]


Train Loss: 0.3666, Acc: 0.8448, F1: 0.8448

Epoch 2/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:42<00:00,  7.35it/s]


Train Loss: 0.2847, Acc: 0.8813, F1: 0.8813

Epoch 3/10
--------------------------------------------------


...

Train Loss: 0.0975, Acc: 0.9634, F1: 0.9634

Epoch 9/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:42<00:00,  7.38it/s]


Train Loss: 0.0856, Acc: 0.9683, F1: 0.9683

Epoch 10/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:42<00:00,  7.34it/s]


Train Loss: 0.0806, Acc: 0.9691, F1: 0.9691

Training completed!


In [15]:
%load_ext tensorboard
%tensorboard --logdir runs/

<IPython.core.display.Javascript object>

In [16]:
import os

save_path = '/kaggle/working/model_v1.pth'
torch.save(model.state_dict(), save_path)
print(f'Модель сохранена: {save_path}')

Модель сохранена: /kaggle/working/model_v1.pth


## 8. Оценка модели на тестовом датасете

In [21]:
# Добавьте логику оценки модели на тестовом датасете, как метрику в tensorboard
# pip install tensorboard
# команда для поднятия
# tensorboard --logdir=my_logs

test_loss, test_acc, test_f1, test_precision, test_recall = validate(
    model, test_loader, criterion, device
)

print(f'Test Loss:      {test_loss:.4f}')
print(f'Test Accuracy:  {test_acc:.4f}')
print(f'Test F1:        {test_f1:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall:    {test_recall:.4f}')

# Логируем тестовые метрики в TensorBoard
# epoch=0 потому что это разовая оценка, а не по эпохам
writer.add_scalar('Loss/test', test_loss, 0)
writer.add_scalar('Accuracy/test', test_acc, 0)
writer.add_scalar('F1/test', test_f1, 0)
writer.add_scalar('Precision/test', test_precision, 0)
writer.add_scalar('Recall/test', test_recall, 0)

writer.close()

Validation: 100%|██████████| 125/125 [00:14<00:00,  8.57it/s]

Test Loss:      0.0852
Test Accuracy:  0.9695
Test F1:        0.9695
Test Precision: 0.9695
Test Recall:    0.9695


In [27]:
!kill 311

In [28]:
%reload_ext tensorboard
%tensorboard --logdir /kaggle/working/runs/

<IPython.core.display.Javascript object>

In [33]:
!pip install tensorboard -q
!python -m tensorboard.main --inspect --logdir=/kaggle/working/runs/

2026-05-16 00:06:20.351709: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778889980.375484     481 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778889980.383173     481 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778889980.402302     481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778889980.402366     481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778889980.402373     481 computation_placer.cc:177] computation placer alr

Всё записалось, но Kaggle не даёт вывести, к сожалению

## 9. Дообучите модель на втором датасете и постройте DVC пайплайн

In [5]:
# Добавьте логику дообучения
# Добавьте PVC пайплайн

# Дообучение выполняется локально через DVC пайплайн
# Скрипт дообучения: scripts/finetune.py
"""
import boto3
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from botocore.client import Config
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path
from tqdm import tqdm
import pandas as pd

BASE_DIR = '/mnt/d/tracking_homework_04/ai-vs-human-generated-dataset-hw'
BUCKET = 'models'
MODEL_IN = 'model_v1.pth'
MODEL_OUT = 'model_v2.pth'
BATCH_SIZE = 32
NUM_EPOCHS = 5
LR = 0.0001

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4')
)

print('Скачивание model_v1.pth из MinIO')
s3.download_file(BUCKET, MODEL_IN, 'model_v1.pth')
print('Скачали!')

class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        file_name = self.data.iloc[idx]['file_name']
        img_path = self.root_dir / file_name
        image = Image.open(img_path).convert('RGB')
        label = int(self.data.iloc[idx]['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file=f'{BASE_DIR}/Train_2/train.csv',
    root_dir=f'{BASE_DIR}/Train_2',
    transform=train_transform
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
print(f'Train_2 dataset size: {len(train_dataset)}')

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load('model_v1.pth', map_location=device))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

writer = SummaryWriter(log_dir='runs/finetune')
writer.add_text('hyperparameters',
    f'lr={LR}, batch_size={BATCH_SIZE}, epochs={NUM_EPOCHS}, optimizer=Adam')

print('Дообучение')
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')

    writer.add_scalar('Loss/finetune', epoch_loss, epoch)
    writer.add_scalar('Accuracy/finetune', epoch_acc, epoch)
    writer.add_scalar('F1/finetune', epoch_f1, epoch)

    print(f'Epoch {epoch+1}: Loss={epoch_loss:.4f}, Acc={epoch_acc:.4f}, F1={epoch_f1:.4f}')

writer.close()

torch.save(model.state_dict(), 'model_v2.pth')
s3.upload_file('model_v2.pth', BUCKET, MODEL_OUT)
print('model_v2.pth загружена в MinIO')

"""

In [ ]:
# Скрипт тестирования: scripts/test_model.py
"""
import boto3
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from botocore.client import Config
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import json

BASE_DIR = '/mnt/d/tracking_homework_04/ai-vs-human-generated-dataset-hw'
BUCKET = 'models'
MODEL_IN = 'model_v2.pth'
BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4')
)

print('Скачивание model_v2.pth из MinIO')
s3.download_file(BUCKET, MODEL_IN, 'model_v2.pth')

class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, folder_override=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.folder_override = folder_override

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        file_name = self.data.iloc[idx]['file_name']
        if self.folder_override:
            file_name = self.folder_override + '/' + file_name.split('/')[-1]
        img_path = self.root_dir / file_name
        image = Image.open(img_path).convert('RGB')
        label = int(self.data.iloc[idx]['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = ImageDataset(
    csv_file=f'{BASE_DIR}/Test_2/test.csv',
    root_dir=f'{BASE_DIR}/Test_2',
    transform=test_transform,
    folder_override='test_data'
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Test_2 dataset size: {len(test_dataset)}')

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load('model_v2.pth', map_location=device))
model = model.to(device)

model.eval()
running_loss = 0.0
all_preds = []
all_labels = []
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss = running_loss / len(test_loader.dataset)
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='weighted')
test_precision = precision_score(all_labels, all_preds, average='weighted')
test_recall = recall_score(all_labels, all_preds, average='weighted')

print(f'Test Loss:      {test_loss:.4f}')
print(f'Test Accuracy:  {test_acc:.4f}')
print(f'Test F1:        {test_f1:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall:    {test_recall:.4f}')

writer = SummaryWriter(log_dir='runs/test_v2')
writer.add_scalar('Loss/test_v2', test_loss, 0)
writer.add_scalar('Accuracy/test_v2', test_acc, 0)
writer.add_scalar('F1/test_v2', test_f1, 0)
writer.add_scalar('Precision/test_v2', test_precision, 0)
writer.add_scalar('Recall/test_v2', test_recall, 0)
writer.close()

metrics = {
    'test_loss': test_loss,
    'test_acc': test_acc,
    'test_f1': test_f1,
    'test_precision': test_precision,
    'test_recall': test_recall
}
with open('metrics_v2.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Метрики сохранены в metrics_v2.json')

"""

### Вывод DVC пайплайна

```
Running stage 'test':
> python scripts/test_model.py
Using device: cpu
Скачивание model_v2.pth из MinIO
Test_2 dataset size: 2000
Testing: 100%|██████████| 63/63 [00:43<00:00,  1.45it/s]
Test Loss:      0.0673
Test Accuracy:  0.9755
Test F1:        0.9755
Test Precision: 0.9756
Test Recall:    0.9755
Метрики сохранены в metrics_v2.json
Updating lock file 'dvc.lock'
```

In [ ]:
# Пайплайн описан в dvc.yaml

"""
stages:
  finetune:
    cmd: python scripts/finetune.py
    deps:
      - scripts/finetune.py
      - ai-vs-human-generated-dataset-hw/Train_2
    outs:
      - model_v2.pth

  test:
    cmd: python scripts/test_model.py
    deps:
      - scripts/test_model.py
      - model_v2.pth
      - ai-vs-human-generated-dataset-hw/Test_2
    metrics:
      - metrics_v2.json:
          cache: false
"""

In [ ]:
# Команда запуска пайплайна локально:
# dvc repro

# Граф пайплайна (dvc dag) - загрузила скриншот в репозиторий:
# +----------+
# | finetune |
# +----------+
#       *
#       *
#       *
#   +------+
#   | test |
#   +------+

## 10. Напишите вывод о полученных результатах

## 10. Сравнение двух версий модели

### Метрики модели v1 (обучена на Train_1, протестирована на Test_1)

| Метрика   | Значение |
|-----------|----------|
| Loss      | 0.0852   |
| Accuracy  | 0.9695   |
| F1        | 0.9695   |
| Precision | 0.9695   |
| Recall    | 0.9695   |

### Метрики модели v2 (дообучена на Train_2, протестирована на Test_2)

| Метрика   | Значение |
|-----------|----------|
| Loss      | 0.0673   |
| Accuracy  | 0.9755   |
| F1        | 0.9755   |
| Precision | 0.9756   |
| Recall    | 0.9755   |

### Вывод

Дообучение на Train_2 улучшило все метрики модели. Accuracy выросла с 96.95% до 97.55%, F1 - с 0.9695 до 0.9755, Loss снизился с 0.0852 до 0.0673.

Улучшение метрик объясняется тем, что модель увидела больше разнообразных данных - Train_2 содержит изображения из другого распределения, что помогло модели лучше обобщать и не переобучаться.

Важно отметить, что улучшение незначительное (0.6% по Accuracy), что ожидаемо, так как базовая модель уже была хорошо обучена (96.95%), и ResNet18 с предобученными весами ImageNet  значально извлекает сильные признаки. Дообучение с меньшим learning rate (0.0001 против 0.001) позволило аккуратно скорректировать веса без потери выученных паттернов.